In [2]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 102.9 MB/s eta 0:00:0000:0100:01


In [3]:
# Dữ liệu 
import pandas as pd
import numpy as np

# Ảnh & PyTorch 
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Tìm kiếm tương đồng
import faiss
from sklearn.metrics.pairwise import cosine_similarity

# Vẽ biểu đồ 
import matplotlib.pyplot as plt
import seaborn as sns

# Tiện ích
import os
from tqdm import tqdm

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', 20)      
plt.rcParams['figure.figsize'] = (10, 5)    
sns.set_style('whitegrid')                  


print('Import thư viện thành công!')
print(f'PyTorch version : {torch.__version__}')
# Kiểm tra máy có GPU không. GPU giúp chạy nhanh hơn CPU rất nhiều.
# Laptop bình thường thường sẽ hiện 'cpu' – không sao, vẫn chạy được.
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

Import thư viện thành công!
PyTorch version : 2.11.0+cu128
Device: cuda


DÙNG GG COLAB MƯỢN GPU T4

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

# ĐƯỜNG DẪN ĐÚNG DỰA TRÊN ẢNH GOOGLE DRIVE CỦA BẠN:
DATA_DIR = '/content/drive/MyDrive/DoAnPython/DuLieuPython'

CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
# Vì 'train_images.zip' đang là file nén, bạn cứ khai báo đường dẫn tệp trước:
IMAGE_ZIP_PATH = os.path.join(DATA_DIR, 'train_images.zip') 

PROCESSED = '../data/processed/'
RESULTS = '../results/'
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)

# Kiểm tra lại đường dẫn
for p in [CSV_PATH, IMAGE_ZIP_PATH]:
    status = 'Đã tìm thấy tệp/thư mục' if os.path.exists(p) else 'Không tìm thấy – kiểm tra lại đường dẫn!'
    print(f'{status}  {p}')

Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv
Đã tìm thấy tệp/thư mục  /content/drive/MyDrive/DoAnPython/DuLieuPython/train_images.zip


TIỀN XỬ LÝ DỮ LIỆU

In [6]:
class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        try:
            img_name = self.df.iloc[idx]['image']
            img_path = os.path.join(self.img_dir, img_name)
            image = self.transform(Image.open(img_path).convert("RGB"))
        except Exception:
            image = torch.zeros(3, 224, 224)
            
        text = str(self.df.iloc[idx]['title'])
        text_token = clip.tokenize([text], truncate=True).squeeze(0)
        return image, text_token

In [7]:
df = pd.read_csv(CSV_PATH)

print(f'Train: {df.shape[0]:,} dòng x {df.shape[1]:,} cột')

df.head()

Train: 34,250 dòng x 5 cột


,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045
2,train_2288590299,000a190fdd715a2a36faed16e2c65df7.jpg,b94cb00ed3e50f78,Maling TTS Canned Pork Luncheon Meat 397 gr,2395904891
3,train_2406599165,00117e4fc239b1b641ff08340b429633.jpg,8514fc58eafea283,Daster Batik Lengan pendek - Motif Acak / Camp...,4093212188
4,train_3369186413,00136d1cf4edede0203f32f05f660588.jpg,a6f319f924ad708c,Nescafe \xc3\x89clair Latte 220ml,3648931069


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34250 entries, 0 to 34249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   posting_id   34250 non-null  object
 1   image        34250 non-null  object
 2   image_phash  34250 non-null  object
 3   title        34250 non-null  object
 4   label_group  34250 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.3+ MB


In [9]:
candidate_df = pd.read_csv(CSV_PATH)

---
## 📊 Bước 1: Phân chia Validation / Test Set
- **Gallery:** Toàn bộ 34,250 ảnh (không thay đổi)
- **Val Set (20%):** Dùng để tune tham số α trong Grid Search
- **Test Set (80%):** Chỉ dùng một lần duy nhất để báo cáo kết quả cuối
- **Không dùng stratify** vì số lượng nhóm (11,014) nhiều hơn số lượng mẫu Val (6,850)

In [ ]:
# ============================================================
# CHIA TẬP VALIDATION / TEST
# Validation (20%): dùng để tuning alpha, pHash threshold
# Test (80%): CHỈ dùng để báo kết quả cuối — không tune!
# ============================================================
from sklearn.model_selection import train_test_split

print('=' * 55)
print('CHIA TẬP VALIDATION / TEST')
print('=' * 55)

# Lọc bỏ nhóm chỉ có 1 ảnh (không đánh giá được)
label_counts_all = candidate_df['label_group'].value_counts()
valid_mask = candidate_df['label_group'].isin(
    label_counts_all[label_counts_all >= 2].index
)
valid_indices = candidate_df[valid_mask].index.tolist()

# Chia val/test — không dùng stratify vì nhiều nhóm chỉ có 2 ảnh
val_idx, test_idx = train_test_split(
    valid_indices,
    test_size=0.8,
    random_state=42
)

print(f'Gallery (toàn bộ)  : {len(candidate_df):,} ảnh')
print(f'Validation set     : {len(val_idx):,} ảnh (20%) → dùng để tuning')
print(f'Test set           : {len(test_idx):,} ảnh (80%) → báo kết quả cuối')

# Lưu lại để tái sử dụng
import json as _json
output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"
split_info = {'val_idx': val_idx, 'test_idx': test_idx}
with open(os.path.join(output_dir, 'split_indices.json'), 'w') as f:
    _json.dump(split_info, f)
print('Đã lưu split_indices.json!')

In [10]:
!pip install sentence-transformers timm
import torch
from sentence_transformers import SentenceTransformer
import timm

device = "cuda" if torch.cuda.is_available() else "cpu"
# Sửa đổi chuỗi tên định danh DINOv3 chuẩn xác trong thư viện timm
dinov3_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0).to(device)
dinov3_model.eval()
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TRÍCH XUẤT VECTO

In [11]:
!unzip -q /content/drive/MyDrive/DoAnPython/DuLieuPython/train_images.zip -d /content/train_images_extracted

In [12]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import timm

csv_path = "/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv"
image_folder = "/content/train_images_extracted/train_images"
candidate_df = pd.read_csv(csv_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Tải mô hình với tên chuẩn hóa
dinov3_model = timm.create_model("vit_base_patch16_dinov3.lvd1689m", pretrained=True, num_classes=0).to(device)
dinov3_model.eval()
minilm_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

# Tự động lấy cấu hình tiền xử lý ảnh chuẩn (kích thước, mean, std) từ chính mô hình DINOv3
data_config = timm.data.resolve_model_data_config(dinov3_model)
dinov3_transform = timm.data.create_transform(**data_config, is_training=False)

class ShopeeDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]["image"]
        img_path = os.path.join(self.img_dir, img_name)
        try:
            image = self.transform(Image.open(img_path).convert("RGB"))
        except Exception:
            # Fallback tạo tensor trống khớp chính xác kích thước yêu cầu của mô hình
            image = torch.zeros(3, data_config['input_size'][1], data_config['input_size'][2])
        text = str(self.df.iloc[idx]["title"])
        return image, text

dataset = ShopeeDataset(candidate_df, image_folder, dinov3_transform)
dataloader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=2)

image_features_list = []
titles_list = []

for images, texts in tqdm(dataloader):
    images = images.to(device)
    with torch.no_grad():
        img_features = dinov3_model(images)
    img_features /= img_features.norm(dim=-1, keepdim=True)
    image_features_list.append(img_features.cpu())
    titles_list.extend(texts)

image_features = torch.cat(image_features_list, dim=0).numpy()
text_features = minilm_model.encode(titles_list, batch_size=128, show_progress_bar=True, convert_to_numpy=True)

output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"
np.save(os.path.join(output_dir, "dinov3_image_features.npy"), image_features)
np.save(os.path.join(output_dir, "multilingual_minilm_text_features.npy"), text_features)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 268/268 [08:52<00:00,  1.99s/it]


Batches:   0%|          | 0/268 [00:00<?, ?it/s]

---
## 🔍 Bước 4: Grid Search tham số α trên Validation Set
- Chỉ dùng `val_query.csv` để tìm α tối ưu
- Công thức fusion: `sim = α * img_sim + (1 - α) * txt_sim`
- Sau đó áp dụng pHash boosting

In [ ]:
# ============================================================
# GRID SEARCH ALPHA TRÊN VALIDATION SET
# ============================================================
print('=' * 55)
print('GRID SEARCH ALPHA (VAL SET)')
print('=' * 55)

import gc
import numpy as np
import pandas as pd
import torch
import os
import json as _json

csv_path   = "/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv"
output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"

candidate_df = pd.read_csv(csv_path)
device = "cuda" if torch.cuda.is_available() else "cpu"

dinov3_image_features = np.load(os.path.join(output_dir, "dinov3_image_features.npy")).astype('float32')
minilm_text_features  = np.load(os.path.join(output_dir, "multilingual_minilm_text_features.npy")).astype('float32')

# Load split indices
with open(os.path.join(output_dir, 'split_indices.json'), 'r') as f:
    split_info = _json.load(f)
val_idx  = split_info['val_idx']
test_idx = split_info['test_idx']

alphas = [0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9]
best_alpha    = 0.7
best_val_map  = -1.0

image_tensor_val = torch.tensor(dinov3_image_features[val_idx]).to(device)
text_tensor_val  = torch.tensor(minilm_text_features[val_idx]).to(device)
image_norm_val   = image_tensor_val / image_tensor_val.norm(dim=-1, keepdim=True)
text_norm_val    = text_tensor_val  / text_tensor_val.norm(dim=-1, keepdim=True)

image_norm_all = torch.tensor(dinov3_image_features).to(device)
text_norm_all  = torch.tensor(minilm_text_features).to(device)
image_norm_all = image_norm_all / image_norm_all.norm(dim=-1, keepdim=True)
text_norm_all  = text_norm_all  / text_norm_all.norm(dim=-1, keepdim=True)

phash_strings = candidate_df["image_phash"].values
phash_ints    = np.array([int(h, 16) for h in phash_strings], dtype=np.uint64)
labels_all    = candidate_df["label_group"].values

for alpha in alphas:
    ap_scores = []
    for q_pos, i in enumerate(val_idx):
        query_label = labels_all[i]
        gt_indices  = np.where(labels_all == query_label)[0]
        gt_indices  = gt_indices[gt_indices != i]
        if len(gt_indices) == 0:
            continue

        # Tính fusion score
        img_sim = torch.matmul(
            image_norm_val[q_pos].unsqueeze(0),
            image_norm_all.T).squeeze().cpu().numpy()
        txt_sim = torch.matmul(
            text_norm_val[q_pos].unsqueeze(0),
            text_norm_all.T).squeeze().cpu().numpy()
        sim = alpha * img_sim + (1 - alpha) * txt_sim

        # pHash boost
        q_phash = phash_ints[i]
        x = q_phash ^ phash_ints
        x = (x & 0x5555555555555555) + ((x >> 1) & 0x5555555555555555)
        x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
        x = (x & 0x0F0F0F0F0F0F0F0F) + ((x >> 4) & 0x0F0F0F0F0F0F0F0F)
        x = (x & 0x00FF00FF00FF00FF) + ((x >> 8) & 0x00FF00FF00FF00FF)
        x = (x & 0x0000FFFF0000FFFF) + ((x >> 16) & 0x0000FFFF0000FFFF)
        ham = ((x & 0x00000000FFFFFFFF) + (x >> 32)).astype(np.uint8)
        sim[ham <= 2] += 0.5
        sim[ham == 0] += 0.5
        sim[i]         = -999  # loại self

        top5 = np.argsort(-sim)[:5]
        is_rel = np.isin(top5, gt_indices)
        hits = np.where(is_rel)[0] + 1
        if len(hits) > 0:
            ap = np.sum(np.arange(1, len(hits)+1) / hits) / min(5, len(gt_indices))
        else:
            ap = 0.0
        ap_scores.append(ap)

    val_map = round(float(np.mean(ap_scores)), 4)
    print(f'  alpha={alpha:.2f} → val mAP@5 = {val_map:.4f}')

    if val_map > best_val_map:
        best_val_map  = val_map
        best_alpha    = alpha

print(f'\n🏆 BEST alpha = {best_alpha}, val mAP@5 = {best_val_map:.4f}')
print(f'Dùng alpha = {best_alpha} để đánh giá trên Test Set')

---
## 🏆 Bước 5: Đánh giá cuối cùng trên Test Set
- Áp dụng `best_alpha` tìm được từ tập Val lên tập Test
- Tính toán Precision@K, Recall@K và mAP@5
- Lưu file `metrics_dinov3_minilm.csv` để đối chiếu

In [ ]:
# ============================================================
# TÍNH METRIC TRÊN TEST SET (80%)
# Dùng best_alpha từ grid search ở trên
# ============================================================
import os
import gc
import json as _json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

csv_path   = "/content/drive/MyDrive/DoAnPython/DuLieuPython/train.csv"
output_dir = "/content/drive/MyDrive/DoAnPython/DuLieuPython"

candidate_df = pd.read_csv(csv_path)
labels       = candidate_df["label_group"].values

dinov3_image_features  = np.load(os.path.join(output_dir, "dinov3_image_features.npy")).astype('float32')
minilm_text_features   = np.load(os.path.join(output_dir, "multilingual_minilm_text_features.npy")).astype('float32')
phash_strings          = candidate_df["image_phash"].values
phash_ints             = np.array([int(h, 16) for h in phash_strings], dtype=np.uint64)

# Load test indices
with open(os.path.join(output_dir, 'split_indices.json'), 'r') as f:
    split_info = _json.load(f)
test_idx = split_info['test_idx']

device = "cuda" if torch.cuda.is_available() else "cpu"

image_norm = torch.tensor(dinov3_image_features).to(device)
text_norm  = torch.tensor(minilm_text_features).to(device)
image_norm = image_norm / image_norm.norm(dim=-1, keepdim=True)
text_norm  = text_norm  / text_norm.norm(dim=-1, keepdim=True)

# Dùng best_alpha từ grid search — nếu chưa chạy grid search thì dùng 0.7
ALPHA = best_alpha if 'best_alpha' in dir() else 0.7
print(f'Đánh giá test set với alpha = {ALPHA}')
print(f'Số query test: {len(test_idx):,}')
print('=' * 55)

ap5_scores = []
p1, r1 = [], []
p3, r3 = [], []
p5, r5 = [], []
p10,r10 = [], []

for i in tqdm(test_idx, desc='Tính metric (Test Set)'):
    query_label = labels[i]
    gt_indices  = np.where(labels == query_label)[0]
    gt_indices  = gt_indices[gt_indices != i]  # loại chính nó
    gt_len      = len(gt_indices)

    # ← QUAN TRỌNG: bỏ qua query không có ảnh liên quan
    # KHÔNG dùng append(0) vì sẽ kéo mAP xuống giả tạo
    if gt_len == 0:
        continue

    # Tính fusion score
    img_sim = torch.matmul(
        image_norm[i].unsqueeze(0), image_norm.T
    ).squeeze().cpu().numpy()
    txt_sim = torch.matmul(
        text_norm[i].unsqueeze(0), text_norm.T
    ).squeeze().cpu().numpy()
    sim = ALPHA * img_sim + (1 - ALPHA) * txt_sim

    # pHash boost (giữ nguyên thuật toán bit-twiddling của Hưng)
    q_phash   = phash_ints[i]
    x         = q_phash ^ phash_ints
    x = (x & 0x5555555555555555) + ((x >> 1) & 0x5555555555555555)
    x = (x & 0x3333333333333333) + ((x >> 2) & 0x3333333333333333)
    x = (x & 0x0F0F0F0F0F0F0F0F) + ((x >> 4) & 0x0F0F0F0F0F0F0F0F)
    x = (x & 0x00FF00FF00FF00FF) + ((x >> 8) & 0x00FF00FF00FF00FF)
    x = (x & 0x0000FFFF0000FFFF) + ((x >> 16) & 0x0000FFFF0000FFFF)
    ham       = ((x & 0x00000000FFFFFFFF) + (x >> 32)).astype(np.uint8)
    sim[ham <= 2] += 0.5
    sim[ham == 0] += 0.5
    sim[i]         = -999  # loại self

    # Lấy top-10
    top10      = np.argsort(-sim)[:10]
    is_rel     = np.isin(top10, gt_indices)

    h1  = is_rel[:1].sum();  p1.append(h1/1);   r1.append(h1/gt_len)
    h3  = is_rel[:3].sum();  p3.append(h3/3);   r3.append(h3/gt_len)
    h5  = is_rel[:5].sum();  p5.append(h5/5);   r5.append(h5/gt_len)
    h10 = is_rel[:10].sum(); p10.append(h10/10); r10.append(h10/gt_len)

    # AP@5
    hits = np.where(is_rel[:5])[0] + 1
    if len(hits) > 0:
        ap = np.sum(np.arange(1, len(hits)+1) / hits) / min(5, gt_len)
    else:
        ap = 0.0
    ap5_scores.append(ap)

# Tổng hợp kết quả
summary_metrics = pd.DataFrame({
    "K"        : [1, 3, 5, 10],
    "Precision": [np.mean(p1), np.mean(p3), np.mean(p5), np.mean(p10)],
    "Recall"   : [np.mean(r1), np.mean(r3), np.mean(r5), np.mean(r10)],
})
summary_metrics[["Precision", "Recall"]] = summary_metrics[["Precision", "Recall"]].round(4)
map_at_5 = round(float(np.mean(ap5_scores)), 4)

print('\n=== KẾT QUẢ CUỐI — DINOv3 + Multilingual MiniLM + pHash ===')
print(f'Tập đánh giá: Test set ({len(test_idx):,} query, 80%)')
print(f'Alpha        : {ALPHA}')
print(summary_metrics.to_string(index=False))
print(f'mAP@5        : {map_at_5}')

# Lưu metric ra CSV để file chung (BTCT_Tuan4_33.ipynb) đọc
summary_metrics['mAP@5']  = map_at_5
summary_metrics['method'] = f'DINOv3 + MultilingualMiniLM + pHash (alpha={ALPHA})'
summary_metrics.to_csv(
    os.path.join(output_dir, 'metrics_dinov3_minilm.csv'), index=False)
print('\nĐã lưu metrics_dinov3_minilm.csv!')

## GHI CHÚ AI HỖ TRỢ

_(Bắt buộc theo yêu cầu thầy)_

| Phần                    | AI hỗ trợ như thế nào                                             | Người kiểm tra    |
| ----------------------- | ----------------------------------------------------------------- | ----------------- |
| Load DINOv3 (timm)      | Cursor gợi ý dùng `resolve_model_data_config` lấy transform chuẩn | Nguyễn Khánh Hưng |
| Bit-twiddling Hamming   | Cursor gợi ý thuật toán tối ưu không cần vòng lặp                 | Nguyễn Khánh Hưng |
| Grid search alpha       | Claude gợi ý quy trình val/test split                             | Nguyễn Khánh Hưng |
| Sửa gt_len=0            | Claude phát hiện lỗi append(0) → đổi thành continue               | Nguyễn Khánh Hưng |
| Đổi multilingual MiniLM | Claude gợi ý model phù hợp Shopee đa ngôn ngữ                     | Nguyễn Khánh Hưng |

## KẾ HOẠCH TUẦN 5

| Nội dung                        | Phương pháp                                    | Mục tiêu                  |
| ------------------------------- | ---------------------------------------------- | ------------------------- |
| Fine-tune DINOv3 trên Shopee    | Triplet Loss với hard negative mining          | Tăng mAP so với zero-shot |
| Thử EfficientNet-B4 thay DINOv3 | Nhẹ hơn, dễ fine-tune hơn                      | So sánh với DINOv3        |
| Tối ưu pHash threshold          | Grid search threshold trên val set             | Tìm threshold tốt nhất    |
| Kết hợp DINOv3 + TF-IDF         | Thay MiniLM bằng TF-IDF (đã chứng minh tốt T3) | Có thể cao hơn MiniLM     |